[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-flightrisk.ipynb)

# Full Project: Employee Flight Risk Score

*AIBits Academy · Machine Learning End To End · Full Project*

Turning HR attrition history into a ranked, actionable "who's likely to leave next" score — and why the highest-accuracy model is the wrong model to ship.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['WA_Fn-UseC_-HR-Employee-Attrition.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> HR teams traditionally learn why an employee left *after* they've already resigned, via an exit interview — too late to intervene. A flight-risk score reframes this as a proactive early-warning system: rank **current** employees by predicted probability of leaving, so managers can prioritize retention conversations, compensation review, or workload changes for the highest-risk, highest-value people before they hand in notice.

> **Dataset & Methodology Note**
>
> The original methodology for this project (Nair, *"A Predictive Analytics Model to Determine the Flight Risk Score of Employees"*) used a **CHAID** (Chi-squared Automatic Interaction Detector) decision tree in R against proprietary company data, reporting 66.7% accuracy. CHAID isn't part of the standard Python/scikit-learn toolkit, and the proprietary dataset isn't available — so this implementation substitutes the well-known **IBM Watson HR Employee Attrition dataset** (1,470 employees, 35 columns, US corporate data, presented as-is) and standard scikit-learn classifiers. The numbers below are freshly computed against this dataset, not the article's figures.

## Step 1 — Load and Prepare

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(df.shape)
print(df['Attrition'].value_counts())

# EmployeeCount, StandardHours, Over18 are constant across all rows — zero predictive value
df = df.drop(columns=['EmployeeCount','StandardHours','Over18','EmployeeNumber'])
y = (df['Attrition']=='Yes').astype(int)
X = df.drop(columns=['Attrition'])

for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col])

237 of 1,470 employees (16.1%) left — a meaningful class imbalance, the same trap covered on the Handling Imbalanced Data page, here in an HR context.

## Step 2 — Stratified Split and a Four-Model Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# 1,029 train / 441 test, both preserving the 16.1% attrition rate

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42).fit(X_train, y_train)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42).fit(X_train, y_train)

for name, model, Xte in [('LogReg',lr,X_test_s),('DecisionTree',dt,X_test),
                       ('RandomForest',rf,X_test),('GradientBoosting',gb,X_test)]:
    pred, proba = model.predict(Xte), model.predict_proba(Xte)[:,1]
    print(f"{name:16s} acc={accuracy_score(y_test,pred):.3f}  prec={precision_score(y_test,pred):.3f}  "
          f"rec={recall_score(y_test,pred):.3f}  f1={f1_score(y_test,pred):.3f}  auc={roc_auc_score(y_test,proba):.3f}")

> **The Best-Accuracy Model Is Useless for This Business Goal**
>
> Logistic Regression posts the highest accuracy (86.6%) and highest precision (0.700) — but its **recall is 0.296**. That means it correctly flags fewer than 3 in 10 employees who actually go on to leave. For a proactive retention program, a model that misses 70% of true flight risks provides almost no early warning at all — the exact same accuracy-vs-recall trap covered in the Crystal Structure full project, here with direct HR consequences.

## Step 3 — Rebalance for Recall, the Metric That Matters

In [ ]:
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(X_train_s, y_train)
pred = lr_balanced.predict(X_test_s)
proba = lr_balanced.predict_proba(X_test_s)[:,1]
print(f"acc={accuracy_score(y_test,pred):.3f}  prec={precision_score(y_test,pred):.3f}  "
      f"rec={recall_score(y_test,pred):.3f}  f1={f1_score(y_test,pred):.3f}  auc={roc_auc_score(y_test,proba):.3f}")

from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, pred))

Adding `class_weight='balanced'` — no resampling, no new data, one parameter — trades 10 points of accuracy (86.6% → 76.4%) for a recall jump from 29.6% to **76.1%**. The confusion matrix shows why: this model correctly flags 54 of the 71 employees who actually left (76%), at the cost of 87 false alarms among the 370 who stayed. AUC barely moves (0.808 → 0.807) — the model's underlying ranking ability was always good; only the decision threshold's trade-off changed.

## Step 4 — What Drives Flight Risk, and Ranking the Highest-Risk Employees

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(6))

risk = X_test.copy()
risk['RiskScore'] = proba
risk['ActualAttrition'] = y_test.values
print(risk.sort_values('RiskScore', ascending=False).head(5)[['Age','MonthlyIncome','OverTime','RiskScore','ActualAttrition']])

All five highest-risk-scored employees in the held-out test set really did leave — the model's top of the ranked list is exactly where a manager running retention conversations would want to start. Compensation (`MonthlyIncome`), `Age`, tenure (`TotalWorkingYears`, `YearsAtCompany`), and `OverTime` dominate the driver list — all directly actionable HR levers, unlike a black-box score with no explanation.

## Visualizing the Drivers and the Trade-Off

Left: the top-6 feature importances from the balanced Random Forest, sorted descending. Right: the confusion matrix for the rebalanced model — 87 unnecessary conversations against 17 missed at-risk employees.

## Key Business Takeaways

- Optimizing for accuracy under 16% class imbalance silently produces a model that misses 70% of actual leavers — the headline metric looked great (86.6%) while the business outcome was nearly worthless.
- `class_weight='balanced'` is a lower-effort first lever than SMOTE oversampling (see Handling Imbalanced Data) and, on this dataset, was sufficient to reach usable recall.
- A "flight risk score" is nothing more exotic than a calibrated predicted probability, ranked highest to lowest — the deployable business artifact is the ranked list, not a single accuracy number.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The attrition rate

Store in `attr_rate` the fraction of employees who left (`y == 1`).

In [ ]:
attr_rate = None   # TODO


In [ ]:
try:
    check("about 16.1%", abs(attr_rate - 237 / 1470) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
attr_rate = float(y.mean())

```

</details>

### Exercise 2 · Medium · Overtime and attrition

`OverTime` was label-encoded (0 = No, 1 = Yes). Store in `rate_by_ot` the attrition rate for each `OverTime` value (a Series indexed 0 and 1) and `ot_higher` = whether overtime workers leave more often.

In [ ]:
rate_by_ot = ot_higher = None   # TODO


In [ ]:
try:
    ref = y.groupby(X["OverTime"]).mean()
    check("rates", abs(rate_by_ot[1] - ref[1]) < 1e-12 and abs(rate_by_ot[0] - ref[0]) < 1e-12)
    check("overtime workers leave more", ot_higher is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
rate_by_ot = y.groupby(X["OverTime"]).mean()
ot_higher = bool(rate_by_ot[1] > rate_by_ot[0])

```

</details>

### Exercise 3 · Stretch · A threshold for 50% precision

Using `proba` from the rebalanced model, find the **lowest** threshold (0.05, 0.10, ..., 0.95) whose precision is at least 0.5. Store it in `thr` and the recall there in `rec_at_thr`.

In [ ]:
thr = rec_at_thr = None   # TODO


In [ ]:
try:
    p_ok = precision_score(y_test, proba >= thr)
    check("precision target met", p_ok >= 0.5)
    check("recall reported", abs(rec_at_thr - recall_score(y_test, proba >= thr)) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
ok = [t for t in np.arange(0.05, 0.96, 0.05) if (proba >= t).sum() > 0 and precision_score(y_test, proba >= t) >= 0.5]
thr = float(min(ok))
rec_at_thr = recall_score(y_test, proba >= thr)

```

HR can choose how many retention conversations it can afford; each threshold is a different capacity-versus-coverage trade-off.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Employee Flight Risk Score**.*